# UL94 Model Comparison (V-0 vs non-V-0)

独立、完整的 nested 5×3 多模型训练与预测代码。默认启用已完成且输入/协议相同的结果缓存。


In [ ]:
# 1. 参数配置与路径设置
from pathlib import Path
import os

# Keep nested CV memory-stable on the shared compute node.
for _thread_var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[_thread_var] = '1'

# 与四个性能的特征筛选保持同一 nested 5×3 划分，避免 XGB 结果不可比。
RANDOM_STATE = 48
OUTER_CV_SPLITS = 5
INNER_CV_SPLITS = 3
N_ITER_SEARCH = 20
# Keep the nested search stable: 2 concurrent CV fits × 4 XGBoost threads.
N_JOBS = 1
XGB_N_JOBS = 2
# 默认复用同一输入和协议产生的完成结果；环境变量可按需强制重算。
# 独立任务默认优先复用同一输入、同一协议的已完成结果。
USE_RESULT_CACHE = True
FORCE_RECOMPUTE = False
CACHE_VERSION = 10  # v10: V-0=1 正类；L1-logistic 使用快速有界收敛

# 兼容 matplotlib_inline 与旧版 matplotlib：必须在第一个 cell 里先执行，避免 cell 输出显示时报错。
try:
    MPLCONFIG_DIR = Path("/tmp/motif_model_compare_mplconfig")
    MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_DIR))
    import matplotlib
    if not hasattr(matplotlib.rcParams, "_get"):
        matplotlib.rcParams._get = matplotlib.rcParams.get
except Exception:
    pass

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (cwd, cwd / "LCMWR", *cwd.parents)
     if candidate.name == "LCMWR" and (candidate / "results").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(f"Cannot locate LCMWR project root from: {cwd}")

RESULTS_DIR = PROJECT_ROOT / "results"
OUTPUT_ROOT = RESULTS_DIR / "model_compare"

# 优先读取各筛选脚本生成的“最终特征 + 样本信息 + 目标列”建模表。
# 如果该文件尚未生成，load_task_data() 会自动用 processed_data + final_features_matrix 兜底合并。
TASK_CONFIG = {
    "LOI": {
        "input_path": RESULTS_DIR / "loi_motif_select" / "best_improved_final_features_with_target.csv",
        "target_column": "LOI",
        "task_type": "regression",
        "processed_data_path": RESULTS_DIR / "loi_motif_select" / "loi_vocab_processed_data.csv",
        "feature_matrix_path": RESULTS_DIR / "loi_motif_select" / "best_improved_final_features_matrix.csv",
    },
    "Tg": {
        "input_path": RESULTS_DIR / "tg_motif_select" / "best_improved_final_features_with_target.csv",
        "target_column": "Tg",
        "task_type": "regression",
        "processed_data_path": RESULTS_DIR / "tg_motif_select" / "tg_vocab_processed_data.csv",
        "feature_matrix_path": RESULTS_DIR / "tg_motif_select" / "best_improved_final_features_matrix.csv",
    },
    "T5": {
        "input_path": RESULTS_DIR / "t5_motif_select" / "best_improved_final_features_with_target.csv",
        "target_column": "T5",
        "task_type": "regression",
        "processed_data_path": RESULTS_DIR / "t5_motif_select" / "t5_vocab_processed_data.csv",
        "feature_matrix_path": RESULTS_DIR / "t5_motif_select" / "best_improved_final_features_matrix.csv",
    },
    "UL94": {
        "input_path": RESULTS_DIR / "ul94_motif_select" / "best_improved_final_features_with_target.csv",
        "target_column": "UL-94",
        "task_type": "classification",
        "processed_data_path": RESULTS_DIR / "ul94_motif_select" / "ul94_vocab_processed_data.csv",
        "feature_matrix_path": RESULTS_DIR / "ul94_motif_select" / "best_improved_final_features_matrix.csv",
    },
}

# 本 notebook 只运行 UL94；不引用或执行其他 notebook。
RUN_TASKS = ['UL94']
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"OUTPUT_ROOT  = {OUTPUT_ROOT}")


In [2]:
# 2. 依赖导入
import json
import os
import random
import time
import warnings
from collections import Counter
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    MPLCONFIG_DIR = OUTPUT_ROOT / "_mplconfig"
    MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_DIR))
except NameError:
    pass

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.ensemble import (
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LogisticRegression, Ridge, RidgeClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, RandomizedSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

try:
    import matplotlib
    if not hasattr(matplotlib.rcParams, "_get"):
        matplotlib.rcParams._get = matplotlib.rcParams.get
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    warnings.warn(f"matplotlib is not available; plots will be skipped: {exc}")

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except Exception as exc:
    LGBMClassifier = LGBMRegressor = None
    LIGHTGBM_AVAILABLE = False
    LIGHTGBM_IMPORT_ERROR = str(exc)

try:
    from xgboost import XGBClassifier, XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBClassifier = XGBRegressor = None
    XGBOOST_AVAILABLE = False
    XGBOOST_IMPORT_ERROR = str(exc)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)


In [3]:
# 3. 数据读取与任务配置
METADATA_COLUMNS = {
    "DOI",
    "SMILES",
    "smiles",
    "co_smiles",
    "smiles1",
    "smiles2",
    "PolymerName",
    "type",
    "polymer_family",
    "P_content",
    "Dripping",
    "composition_mode",
    "co_mode",
    "blend_mode",
}

SAMPLE_ID_COLUMNS = [
    "DOI",
    "type",
    "polymer_family",
    "PolymerName",
]


def load_feature_target_table(config: Dict[str, Any]) -> Tuple[pd.DataFrame, str]:
    """读取带 target 的建模表；若尚未保存，则按行号兜底合并 processed_data 与最终特征矩阵。"""
    input_path = Path(config["input_path"])
    if input_path.exists():
        return pd.read_csv(input_path), str(input_path)

    processed_path = Path(config.get("processed_data_path", ""))
    feature_path = Path(config.get("feature_matrix_path", ""))
    if not processed_path.exists() or not feature_path.exists():
        raise FileNotFoundError(
            f"Input matrix not found: {input_path}. Fallback files also missing: "
            f"processed_data_path={processed_path}, feature_matrix_path={feature_path}"
        )

    processed = pd.read_csv(processed_path).reset_index(drop=True)
    features = pd.read_csv(feature_path).reset_index(drop=True)
    if len(processed) != len(features):
        raise ValueError(
            f"Cannot align fallback files by row: {processed_path} has {len(processed)} rows, "
            f"but {feature_path} has {len(features)} rows."
        )
    target_column = config["target_column"]
    meta_cols = [col for col in SAMPLE_ID_COLUMNS + [target_column, "P_content", "Dripping"] if col in processed.columns]
    merged = pd.concat([processed[meta_cols], features], axis=1)
    return merged, f"fallback:{processed_path}+{feature_path}"


def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        value = float(obj)
        return None if np.isnan(value) else value
    if isinstance(obj, np.ndarray):
        return to_jsonable(obj.tolist())
    if pd.isna(obj) if not isinstance(obj, (list, tuple, dict, np.ndarray)) else False:
        return None
    return obj


def write_json(path: Path, data: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(to_jsonable(data), f, ensure_ascii=False, indent=2)


def read_json(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def file_signature(path: Path) -> Dict[str, Any]:
    path = Path(path)
    if not path.exists():
        return {"path": str(path), "exists": False}
    stat = path.stat()
    return {
        "path": str(path),
        "exists": True,
        "size": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
    }


def task_input_signature(config: Dict[str, Any]) -> Dict[str, Any]:
    signature = {"input_path": file_signature(Path(config["input_path"]))}
    for key in ("processed_data_path", "feature_matrix_path"):
        if key in config:
            signature[key] = file_signature(Path(config[key]))
    return signature


def cache_expected_config(task_name: str, config: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "cache_version": CACHE_VERSION,
        "task_name": task_name,
        "input_path": str(config["input_path"]),
        "target_column": config["target_column"],
        "task_type": config["task_type"],
        "outer_cv_splits_requested": OUTER_CV_SPLITS,
        "inner_cv_splits_requested": INNER_CV_SPLITS,
        "n_iter_search": N_ITER_SEARCH,
        "random_state": RANDOM_STATE,
        "model_order": MODEL_ORDER,
        "classification_inner_scoring": "roc_auc",
        "input_signature": task_input_signature(config),
    }


def cache_matches(saved_config: Dict[str, Any], expected_config: Dict[str, Any]) -> bool:
    if saved_config.get("status") != "success":
        return False
    for key, expected_value in expected_config.items():
        if saved_config.get(key) != expected_value:
            return False
    return True


def load_cached_task_result(task_name: str, config: Dict[str, Any], output_dir: Path) -> Optional[Dict[str, Any]]:
    if not USE_RESULT_CACHE or FORCE_RECOMPUTE:
        return None

    required_files = {
        "run_config": output_dir / "run_config.json",
        "summary": output_dir / "model_performance_summary.csv",
        "predictions": output_dir / "cv_predictions.csv",
        "fold_metrics": output_dir / "fold_metrics.csv",
        "best_params": output_dir / "best_params.json",
    }
    if not all(path.exists() and path.stat().st_size > 0 for path in required_files.values()):
        return None

    try:
        saved_config = read_json(required_files["run_config"])
        expected_config = cache_expected_config(task_name, config)
        if not cache_matches(saved_config, expected_config):
            return None
        summary_df = pd.read_csv(required_files["summary"])
    except Exception as exc:
        print(f"[{task_name}] cache ignored: {exc}")
        return None

    print(f"[{task_name}] using cached results from {output_dir}")
    return {
        "task": task_name,
        "status": "cached",
        "output_dir": output_dir,
        "error_message": "",
        "summary": summary_df,
    }


def load_task_data(task_name: str, config: Dict[str, Any]) -> Tuple[pd.DataFrame, pd.Series, Dict[str, Any]]:
    input_path = Path(config["input_path"])
    target_column = config["target_column"]
    task_type = config["task_type"]

    df, resolved_input = load_feature_target_table(config)
    if target_column not in df.columns:
        raise ValueError(
            f"Target column '{target_column}' is not available after loading {resolved_input}. "
            "Please regenerate the corresponding *_motif_select notebook or update TASK_CONFIG."
        )

    before_drop = len(df)
    df = df.loc[df[target_column].notna()].copy()
    dropped_target_na = before_drop - len(df)

    candidate_features = df.drop(columns=[target_column])
    numeric_features = candidate_features.select_dtypes(include=[np.number]).columns.tolist()
    feature_columns = [c for c in numeric_features if c not in METADATA_COLUMNS]

    if not feature_columns:
        raise ValueError(f"No numeric feature columns found for task {task_name} in {input_path}")

    # 筛选阶段与 seed 扫描都以 float32 输入 XGBoost(hist)。
    # 保持 dtype 一致，否则 hist 分箱与内层最优超参数会发生漂移。
    X = df[feature_columns].astype(np.float32, copy=False).copy()
    y = df[target_column].copy()

    metadata = {
        "task_name": task_name,
        "input_path": input_path,
        "resolved_input": resolved_input,
        "target_column": target_column,
        "task_type": task_type,
        "n_samples": int(len(X)),
        "n_features": int(X.shape[1]),
        "feature_columns": feature_columns,
        "dropped_target_na": int(dropped_target_na),
        "feature_dtype": str(X.dtypes.iloc[0]) if not X.empty else "float32",
        "excluded_non_numeric_columns": [c for c in candidate_features.columns if c not in numeric_features],
        "excluded_numeric_metadata_columns": [c for c in numeric_features if c in METADATA_COLUMNS],
    }
    sample_metadata = df[[col for col in SAMPLE_ID_COLUMNS if col in df.columns]].copy()
    metadata["sample_metadata_columns"] = sample_metadata.columns.tolist()
    return X, y, metadata, sample_metadata


def make_cv(task_type: str, y: pd.Series, n_splits: int, random_state: int):
    if task_type == "classification":
        counts = pd.Series(y).value_counts(dropna=False)
        max_splits = int(min(n_splits, counts.min()))
        if max_splits < 2:
            raise ValueError("Classification CV needs at least 2 samples in every class.")
        return StratifiedKFold(n_splits=max_splits, shuffle=True, random_state=random_state)

    max_splits = min(n_splits, len(y))
    if max_splits < 2:
        raise ValueError("Regression CV needs at least 2 samples.")
    return KFold(n_splits=max_splits, shuffle=True, random_state=random_state)


In [4]:
# 4. 模型与参数空间定义
STANDARDIZED_MODELS = {"MLP", "SVR/SVM", "Ridge", "Lasso", "KNN"}
MODEL_ORDER = [
    "MLP",
    "SVR/SVM",
    "Ridge",
    "Lasso",
    "KNN",
    "DecisionTree",
    "LightGBM",
    "GradientBoosting",
    "RandomForest",
    "ExtraTrees",
    "XGBoost",
]


@dataclass
class ModelSpec:
    name: str
    estimator: Any
    param_distributions: Dict[str, List[Any]]
    requires_scaling: bool
    available: bool = True
    unavailable_reason: str = ""
    fallback_params: Optional[Dict[str, Any]] = None


def get_model_specs(task_type: str, n_classes: Optional[int] = None) -> List[ModelSpec]:
    if task_type == "regression":
        specs = [
            ModelSpec(
                "MLP",
                MLPRegressor(max_iter=5000, early_stopping=True, n_iter_no_change=100, random_state=RANDOM_STATE),
                {
                    "model__hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64)],
                    "model__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
                    "model__learning_rate_init": [1e-5, 1e-4, 3e-4, 1e-3],
                },
                True,
            ),
            ModelSpec(
                "SVR/SVM",
                SVR(),
                {
                    "model__C": [0.1, 1, 10, 100],
                    "model__gamma": ["scale", "auto", 0.01, 0.1],
                    "model__epsilon": [0.01, 0.05, 0.1, 0.2],
                    "model__kernel": ["rbf"],
                },
                True,
            ),
            ModelSpec("Ridge", Ridge(solver="lsqr", max_iter=100000, tol=1e-5, random_state=RANDOM_STATE), {"model__alpha": [0.1, 1, 10, 100, 1000, 10000]}, True),
            ModelSpec(
                "Lasso",
                Lasso(max_iter=100000, tol=1e-4, random_state=RANDOM_STATE),
                {"model__alpha": [1e-3, 1e-2, 0.1, 1.0, 3.0, 10.0]},
                True,
            ),
            ModelSpec(
                "KNN",
                KNeighborsRegressor(n_jobs=N_JOBS),
                {
                    "model__n_neighbors": [3, 5, 7, 11, 15],
                    "model__weights": ["uniform", "distance"],
                    "model__p": [1, 2],
                },
                True,
            ),
            ModelSpec(
                "DecisionTree",
                DecisionTreeRegressor(random_state=RANDOM_STATE),
                {
                    "model__max_depth": [None, 3, 5, 10, 20],
                    "model__min_samples_split": [2, 5, 10],
                    "model__min_samples_leaf": [1, 2, 4],
                },
                False,
            ),
            ModelSpec(
                "LightGBM",
                LGBMRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS, verbosity=-1) if LIGHTGBM_AVAILABLE else None,
                {
                    "model__n_estimators": [100, 300, 600],
                    "model__learning_rate": [0.01, 0.05, 0.1],
                    "model__num_leaves": [15, 31, 63],
                    "model__max_depth": [-1, 5, 10],
                },
                False,
                LIGHTGBM_AVAILABLE,
                "lightgbm is not installed" if not LIGHTGBM_AVAILABLE else "",
            ),
            ModelSpec(
                "GradientBoosting",
                GradientBoostingRegressor(random_state=RANDOM_STATE),
                {
                    "model__n_estimators": [100, 300, 500],
                    "model__learning_rate": [0.01, 0.05, 0.1],
                    "model__max_depth": [2, 3, 5],
                    "model__subsample": [0.7, 0.9, 1.0],
                },
                False,
            ),
            ModelSpec(
                "RandomForest",
                RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS),
                {
                    "model__n_estimators": [100, 300, 500],
                    "model__max_depth": [None, 5, 10, 20],
                    "model__min_samples_split": [2, 5, 10],
                    "model__min_samples_leaf": [1, 2, 4],
                },
                False,
            ),
            ModelSpec(
                "ExtraTrees",
                ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS),
                {
                    "model__n_estimators": [100, 300, 500],
                    "model__max_depth": [None, 5, 10, 20],
                    "model__min_samples_split": [2, 5, 10],
                    "model__min_samples_leaf": [1, 2, 4],
                },
                False,
            ),
            ModelSpec(
                "XGBoost",
                XGBRegressor(random_state=RANDOM_STATE, n_jobs=XGB_N_JOBS, verbosity=0, objective="reg:squarederror", tree_method="hist") if XGBOOST_AVAILABLE else None,
                {
                    "model__n_estimators": [100, 300, 600],
                    "model__learning_rate": [0.01, 0.05, 0.1],
                    "model__max_depth": [2, 3, 5, 7],
                    "model__subsample": [0.7, 0.9, 1.0],
                    "model__colsample_bytree": [0.7, 0.9, 1.0],
                },
                False,
                XGBOOST_AVAILABLE,
                "xgboost is not installed" if not XGBOOST_AVAILABLE else "",
                {"model__n_estimators": 200, "model__max_depth": 5, "model__learning_rate": 0.1},
            ),
        ]
        return specs

    specs = [
        ModelSpec(
            "MLP",
            MLPClassifier(max_iter=5000, early_stopping=True, n_iter_no_change=100, random_state=RANDOM_STATE),
            {
                "model__hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64)],
                "model__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
                "model__learning_rate_init": [1e-5, 1e-4, 3e-4, 1e-3],
            },
            True,
        ),
        ModelSpec(
            "SVR/SVM",
            SVC(probability=True, random_state=RANDOM_STATE),
            {
                "model__C": [0.1, 1, 10, 100],
                "model__gamma": ["scale", "auto", 0.01, 0.1],
                "model__kernel": ["rbf"],
            },
            True,
        ),
        ModelSpec(
            "Ridge",
            RidgeClassifier(solver="lsqr", max_iter=100000, tol=1e-5),
            {"model__alpha": [0.1, 1, 10, 100, 1000, 10000]},
            True,
        ),
        ModelSpec(
            "Lasso",
            LogisticRegression(penalty="l1", solver="liblinear", max_iter=300, tol=1e-3, random_state=RANDOM_STATE),
            {"model__C": [0.01, 0.1, 1, 10, 100]},
            True,
        ),
        ModelSpec(
            "KNN",
            KNeighborsClassifier(n_jobs=N_JOBS),
            {
                "model__n_neighbors": [3, 5, 7, 11, 15],
                "model__weights": ["uniform", "distance"],
                "model__p": [1, 2],
            },
            True,
        ),
        ModelSpec(
            "DecisionTree",
            DecisionTreeClassifier(random_state=RANDOM_STATE),
            {
                "model__max_depth": [None, 3, 5, 10, 20],
                "model__min_samples_split": [2, 5, 10],
                "model__min_samples_leaf": [1, 2, 4],
            },
            False,
        ),
        ModelSpec(
            "LightGBM",
            LGBMClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS, verbosity=-1) if LIGHTGBM_AVAILABLE else None,
            {
                "model__n_estimators": [100, 300, 600],
                "model__learning_rate": [0.01, 0.05, 0.1],
                "model__num_leaves": [15, 31, 63],
                "model__max_depth": [-1, 5, 10],
            },
            False,
            LIGHTGBM_AVAILABLE,
            "lightgbm is not installed" if not LIGHTGBM_AVAILABLE else "",
        ),
        ModelSpec(
            "GradientBoosting",
            GradientBoostingClassifier(random_state=RANDOM_STATE),
            {
                "model__n_estimators": [100, 300, 500],
                "model__learning_rate": [0.01, 0.05, 0.1],
                "model__max_depth": [2, 3, 5],
                "model__subsample": [0.7, 0.9, 1.0],
            },
            False,
        ),
        ModelSpec(
            "RandomForest",
            RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS),
            {
                "model__n_estimators": [100, 300, 500],
                "model__max_depth": [None, 5, 10, 20],
                "model__min_samples_split": [2, 5, 10],
                "model__min_samples_leaf": [1, 2, 4],
            },
            False,
        ),
        ModelSpec(
            "ExtraTrees",
            ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS),
            {
                "model__n_estimators": [100, 300, 500],
                "model__max_depth": [None, 5, 10, 20],
                "model__min_samples_split": [2, 5, 10],
                "model__min_samples_leaf": [1, 2, 4],
            },
            False,
        ),
        ModelSpec(
            "XGBoost",
            XGBClassifier(random_state=RANDOM_STATE, n_jobs=XGB_N_JOBS, verbosity=0, eval_metric="logloss", tree_method="hist") if XGBOOST_AVAILABLE else None,
            {
                "model__n_estimators": [100, 300, 600],
                "model__learning_rate": [0.01, 0.05, 0.1],
                "model__max_depth": [2, 3, 5, 7],
                "model__subsample": [0.7, 0.9, 1.0],
                "model__colsample_bytree": [0.7, 0.9, 1.0],
            },
            False,
            XGBOOST_AVAILABLE,
            "xgboost is not installed" if not XGBOOST_AVAILABLE else "",
            {"model__n_estimators": 200, "model__max_depth": 5, "model__learning_rate": 0.1},
        ),
    ]
    return specs


In [5]:
# 5. Pipeline 构建函数

def build_pipeline(spec: ModelSpec) -> Pipeline:
    steps = [("imputer", SimpleImputer(strategy="constant", fill_value=0))]
    if spec.requires_scaling:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", clone(spec.estimator)))
    return Pipeline(steps)


def get_search_scoring(task_type: str) -> str:
    return "r2" if task_type == "regression" else "roc_auc"


def get_inner_cv(task_type: str, y_train: pd.Series):
    return make_cv(task_type, y_train, INNER_CV_SPLITS, RANDOM_STATE)


In [6]:
# 6. 单模型评估函数

def regression_metrics(y_true, y_pred) -> Dict[str, float]:
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": mean_absolute_error(y_true, y_pred),
    }


def get_continuous_scores(estimator: Pipeline, X_valid: pd.DataFrame):
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X_valid)
    if hasattr(estimator, "decision_function"):
        return estimator.decision_function(X_valid)
    return None


def safe_roc_auc(y_true, scores, n_classes: int) -> float:
    if scores is None:
        return np.nan
    try:
        if n_classes == 2:
            if getattr(scores, "ndim", 1) == 2:
                scores = scores[:, 1]
            return roc_auc_score(y_true, scores)
        return roc_auc_score(y_true, scores, multi_class="ovr", average="macro")
    except Exception:
        return np.nan


def classification_metrics(y_true, y_pred, scores, n_classes: int) -> Dict[str, float]:
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "ROC-AUC": safe_roc_auc(y_true, scores, n_classes),
    }


def aggregate_fold_metrics(fold_metrics: List[Dict[str, Any]], task_type: str) -> Dict[str, Any]:
    metric_names = ["R2", "RMSE", "MAE"] if task_type == "regression" else [
        "Accuracy",
        "Balanced Accuracy",
        "Precision macro",
        "Recall macro",
        "F1 macro",
        "ROC-AUC",
    ]
    summary = {}
    frame = pd.DataFrame(fold_metrics)
    for metric in metric_names:
        values = pd.to_numeric(frame[metric], errors="coerce") if metric in frame else pd.Series(dtype=float)
        summary[f"{metric}_mean"] = values.mean(skipna=True)
        summary[f"{metric}_std"] = values.std(skipna=True, ddof=1)
    return summary


def evaluate_single_model(
    task_name: str,
    spec: ModelSpec,
    X: pd.DataFrame,
    y_model: pd.Series,
    y_display: pd.Series,
    outer_cv,
    task_type: str,
    class_labels: Optional[List[Any]] = None,
) -> Dict[str, Any]:
    started = time.time()
    n_samples = len(X)
    predictions = pd.Series(index=X.index, dtype="object" if task_type == "classification" else "float64")
    fold_assignments = pd.Series(index=X.index, dtype="float64")

    if not spec.available:
        return {
            "model": spec.name,
            "status": "skipped",
            "error_message": spec.unavailable_reason,
            "elapsed_seconds": 0.0,
            "predictions": predictions,
            "fold_assignments": fold_assignments,
            "fold_metrics": [],
            "best_params_by_fold": [],
            "summary_metrics": {},
        }

    fold_metrics = []
    best_params_by_fold = []

    try:
        for fold, (train_idx, valid_idx) in enumerate(outer_cv.split(X, y_model), start=1):
            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y_model.iloc[train_idx], y_model.iloc[valid_idx]

            pipeline = build_pipeline(spec)
            selection_method = "random_search"
            search_error = ""
            try:
                search = RandomizedSearchCV(
                    estimator=pipeline,
                    param_distributions=spec.param_distributions,
                    n_iter=N_ITER_SEARCH,
                    scoring=get_search_scoring(task_type),
                    cv=get_inner_cv(task_type, y_train),
                    random_state=RANDOM_STATE,
                    n_jobs=N_JOBS,
                    refit=True,
                    error_score="raise",
                )
                search.fit(X_train, y_train)
                best_estimator = search.best_estimator_
                best_score_inner_cv = search.best_score_
                selected_params = search.best_params_
            except Exception as search_exc:
                if not spec.fallback_params:
                    raise
                selection_method = "feature_selection_defaults_fallback"
                search_error = repr(search_exc)
                selected_params = dict(spec.fallback_params)
                best_score_inner_cv = np.nan
                best_estimator = build_pipeline(spec)
                best_estimator.set_params(**selected_params)
                best_estimator.fit(X_train, y_train)
                print(f"[{task_name}] {spec.name} fold {fold}: random search failed; using feature-selection defaults. {search_error}", flush=True)

            y_pred = best_estimator.predict(X_valid)

            if task_type == "regression":
                fold_result = regression_metrics(y_valid, y_pred)
                predictions.iloc[valid_idx] = y_pred.astype(float)
                fold_assignments.iloc[valid_idx] = fold
            else:
                scores = get_continuous_scores(best_estimator, X_valid)
                fold_result = classification_metrics(y_valid, y_pred, scores, len(class_labels or []))
                display_pred = pd.Series(y_pred).map(lambda v: class_labels[int(v)] if class_labels is not None else v).to_numpy()
                predictions.iloc[valid_idx] = display_pred
                fold_assignments.iloc[valid_idx] = fold

            fold_result.update({
                "task": task_name,
                "model": spec.name,
                "fold": fold,
                "best_score_inner_cv": best_score_inner_cv,
                "selection_method": selection_method,
                "search_error": search_error,
            })
            fold_metrics.append(fold_result)
            best_params_by_fold.append({
                "fold": fold,
                "best_params": selected_params,
                "best_score_inner_cv": best_score_inner_cv,
                "selection_method": selection_method,
                "search_error": search_error,
            })

        elapsed = time.time() - started
        summary_metrics = aggregate_fold_metrics(fold_metrics, task_type)
        representative = max(best_params_by_fold, key=lambda item: item.get("best_score_inner_cv", -np.inf) if np.isfinite(item.get("best_score_inner_cv", np.nan)) else -np.inf) if best_params_by_fold else {}
        return {
            "model": spec.name,
            "status": "success",
            "error_message": "",
            "elapsed_seconds": elapsed,
            "predictions": predictions,
            "fold_assignments": fold_assignments,
            "fold_metrics": fold_metrics,
            "best_params_by_fold": best_params_by_fold,
            "representative_best_params": representative.get("best_params", {}),
            "summary_metrics": summary_metrics,
        }
    except Exception as exc:
        elapsed = time.time() - started
        return {
            "model": spec.name,
            "status": "failed",
            "error_message": repr(exc),
            "elapsed_seconds": elapsed,
            "predictions": predictions,
            "fold_assignments": fold_assignments,
            "fold_metrics": fold_metrics,
            "best_params_by_fold": best_params_by_fold,
            "representative_best_params": {},
            "summary_metrics": aggregate_fold_metrics(fold_metrics, task_type) if fold_metrics else {},
        }


In [7]:
# 7. 单任务运行函数


def make_performance_plot(summary_df: pd.DataFrame, task_type: str, output_dir: Path) -> None:
    if plt is None or summary_df.empty:
        return

    metric = "R2_mean" if task_type == "regression" else "ROC-AUC_mean"
    if metric not in summary_df.columns:
        metric = "ROC-AUC_mean" if "ROC-AUC_mean" in summary_df.columns else ("Balanced Accuracy_mean" if "Balanced Accuracy_mean" in summary_df.columns else None)
    if metric is None:
        return

    plot_df = summary_df.loc[summary_df["status"].eq("success")].copy()
    if plot_df.empty:
        return
    plot_df[metric] = pd.to_numeric(plot_df[metric], errors="coerce")
    plot_df = plot_df.sort_values(metric, ascending=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(plot_df["model"], plot_df[metric], color="#4C78A8")
    ax.set_ylabel(metric.replace("_mean", ""))
    ax.set_xlabel("")
    ax.set_title(f"{plot_df['task'].iloc[0]} model comparison")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(output_dir / "model_performance_barplot.png", dpi=200)
    plt.close(fig)


def run_task(task_name: str, config: Dict[str, Any]) -> Dict[str, Any]:
    task_output_dir = OUTPUT_ROOT / task_name
    task_output_dir.mkdir(parents=True, exist_ok=True)

    run_config_base = cache_expected_config(task_name, config)
    cached_result = load_cached_task_result(task_name, config, task_output_dir)
    if cached_result is not None:
        return cached_result

    try:
        X, y_raw, metadata, sample_metadata = load_task_data(task_name, config)
        task_type = metadata["task_type"]

        if task_type == "classification":
            # UL94 以 V-0（更好的阻燃性能）为正类；避免 LabelEncoder 字典序改变 AUC 的方向。
            y_display = y_raw.astype(str)
            expected_labels = {'V-0', 'non-V-0'}
            observed_labels = set(y_display.unique())
            if observed_labels != expected_labels:
                raise ValueError(f'UL94 标签必须为 {sorted(expected_labels)}，当前为 {sorted(observed_labels)}')
            y_model = pd.Series((y_display == 'V-0').astype(int), index=y_raw.index, name=y_raw.name)
            class_labels = ['non-V-0', 'V-0']  # 数值编码：0=non-V-0，1=V-0（正类）
        else:
            y_model = pd.to_numeric(y_raw, errors="coerce")
            valid_target = y_model.notna()
            X = X.loc[valid_target]
            y_model = y_model.loc[valid_target]
            y_display = y_model.copy()
            class_labels = None
            metadata["n_samples"] = int(len(X))

        outer_cv = make_cv(task_type, y_model, OUTER_CV_SPLITS, RANDOM_STATE)
        actual_outer_splits = outer_cv.get_n_splits(X, y_model)
        model_specs = get_model_specs(task_type, len(class_labels) if class_labels else None)

        predictions_df = sample_metadata.loc[X.index].reset_index(drop=True).copy()
        predictions_df.insert(0, "sample_index", X.index.to_numpy())
        predictions_df["true_value"] = y_display.loc[X.index].to_numpy()
        summary_rows = []
        all_fold_metrics = []
        best_params = {}

        for spec in model_specs:
            print(f"[{task_name}] running {spec.name} ...", flush=True)
            result = evaluate_single_model(
                task_name=task_name,
                spec=spec,
                X=X,
                y_model=y_model.loc[X.index],
                y_display=y_display.loc[X.index],
                outer_cv=outer_cv,
                task_type=task_type,
                class_labels=class_labels,
            )

            predictions_df[spec.name] = result["predictions"].reindex(X.index).to_numpy()
            predictions_df[f"{spec.name}_fold"] = result["fold_assignments"].reindex(X.index).to_numpy()
            all_fold_metrics.extend(result["fold_metrics"])

            summary_row = {
                "task": task_name,
                "model": spec.name,
                "status": result["status"],
                "error_message": result["error_message"],
                "n_samples": int(X.shape[0]),
                "n_features": int(X.shape[1]),
                "target_column": metadata["target_column"],
                "task_type": task_type,
                "elapsed_seconds": result["elapsed_seconds"],
            }
            summary_row.update(result["summary_metrics"])
            summary_rows.append(summary_row)

            best_params[spec.name] = {
                "status": result["status"],
                "error_message": result["error_message"],
                "best_params": result.get("representative_best_params", {}),
                "best_params_by_fold": result.get("best_params_by_fold", []),
            }

        summary_df = pd.DataFrame(summary_rows)
        fold_metrics_df = pd.DataFrame(all_fold_metrics)

        summary_df.to_csv(task_output_dir / "model_performance_summary.csv", index=False, encoding="utf-8-sig")
        predictions_df.to_csv(task_output_dir / "cv_predictions.csv", index=False, encoding="utf-8-sig")
        fold_metrics_df.to_csv(task_output_dir / "fold_metrics.csv", index=False, encoding="utf-8-sig")
        write_json(task_output_dir / "best_params.json", best_params)

        run_config = {
            **run_config_base,
            **metadata,
            "outer_cv_splits_actual": actual_outer_splits,
            "class_labels": class_labels,
            "status": "success",
        }
        write_json(task_output_dir / "run_config.json", run_config)
        try:
            make_performance_plot(summary_df, task_type, task_output_dir)
        except Exception as plot_error:
            print(f"[{task_name}] plot skipped: {plot_error}")

        return {"task": task_name, "status": "success", "output_dir": task_output_dir, "summary": summary_df}

    except Exception as exc:
        error_config = {**run_config_base, "status": "failed", "error_message": repr(exc)}
        write_json(task_output_dir / "run_config.json", error_config)
        failed_summary = pd.DataFrame([{
            "task": task_name,
            "model": "ALL",
            "status": "failed",
            "error_message": repr(exc),
            "n_samples": np.nan,
            "n_features": np.nan,
            "target_column": config["target_column"],
            "task_type": config["task_type"],
            "elapsed_seconds": 0.0,
        }])
        failed_summary.to_csv(task_output_dir / "model_performance_summary.csv", index=False, encoding="utf-8-sig")
        write_json(task_output_dir / "best_params.json", {})
        pd.DataFrame().to_csv(task_output_dir / "cv_predictions.csv", index=False, encoding="utf-8-sig")
        pd.DataFrame().to_csv(task_output_dir / "fold_metrics.csv", index=False, encoding="utf-8-sig")
        print(f"[{task_name}] failed: {exc}")
        return {"task": task_name, "status": "failed", "output_dir": task_output_dir, "error_message": repr(exc), "summary": failed_summary}


In [ ]:
# 8. 四个任务批量运行
all_results = []
for task_name in RUN_TASKS:
    result = run_task(task_name, TASK_CONFIG[task_name])
    all_results.append(result)

batch_status = pd.DataFrame([
    {
        "task": item["task"],
        "status": item["status"],
        "output_dir": str(item["output_dir"]),
        "error_message": item.get("error_message", ""),
    }
    for item in all_results
])
batch_status.to_csv(OUTPUT_ROOT / "batch_status.csv", index=False, encoding="utf-8-sig")
cached_count = int(batch_status["status"].eq("cached").sum())
if cached_count:
    print(f"使用缓存的任务数: {cached_count}")
batch_status


In [9]:
# 9. 结果汇总与可视化

summary_files = sorted(OUTPUT_ROOT.glob("*/model_performance_summary.csv"))
if summary_files:
    combined_summary = pd.concat([pd.read_csv(p) for p in summary_files], ignore_index=True)
    combined_summary.to_csv(OUTPUT_ROOT / "all_tasks_model_performance_summary.csv", index=False, encoding="utf-8-sig")
    display(combined_summary)
else:
    combined_summary = pd.DataFrame()
    print("No task summaries found.")


,task,model,status,error_message,n_samples,n_features,target_column,task_type,elapsed_seconds,R2_mean,...,Balanced Accuracy_mean,Balanced Accuracy_std,Precision macro_mean,Precision macro_std,Recall macro_mean,Recall macro_std,F1 macro_mean,F1 macro_std,ROC-AUC_mean,ROC-AUC_std
0,LOI,MLP,success,NaN,948,187,LOI,regression,173.484216,0.627962,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,LOI,SVR/SVM,success,NaN,948,187,LOI,regression,6.788672,0.715575,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LOI,Ridge,success,NaN,948,187,LOI,regression,1.258063,0.622227,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,LOI,Lasso,success,NaN,948,187,LOI,regression,27.345456,0.626916,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,LOI,KNN,success,NaN,948,187,LOI,regression,2.439533,0.683597,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,LOI,DecisionTree,success,NaN,948,187,LOI,regression,1.997698,0.614332,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,LOI,LightGBM,success,NaN,948,187,LOI,regression,21.130097,0.734932,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,LOI,GradientBoosting,success,NaN,948,187,LOI,regression,124.053016,0.767401,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,LOI,RandomForest,success,NaN,948,187,LOI,regression,180.349741,0.755528,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,LOI,ExtraTrees,success,NaN,948,187,LOI,regression,125.912214,0.784805,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
